In [ ]:
# ================================
# CELL 1 — Install
# ================================
!pip install rank_bm25 \
sentence-transformers -q
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("✅ Done!")

In [ ]:
# ================================
# CELL 2 — Load All Files
# ================================
import pickle
import torch
from google.colab import files

print("Upload all pkl and pt files...")
files.upload()

with open('processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

statute_texts   = data['statute_texts']
statute_ids     = data['statute_ids']
precedent_texts = data['precedent_texts']
precedent_ids   = data['precedent_ids']
query_texts     = data['query_texts']
query_ids       = data['query_ids']
gold_statutes   = data['gold_statutes']
gold_precedents = data['gold_precedents']

with open('bm25_results.pkl', 'rb') as f:
    bm25_data = pickle.load(f)

lsr_results = bm25_data['lsr_results']
pcr_results = bm25_data['pcr_results']

statute_embeddings   = torch.load(
    'statute_embeddings.pt')
precedent_embeddings = torch.load(
    'precedent_embeddings.pt')
query_embeddings     = torch.load(
    'query_embeddings.pt')

print("✅ Everything loaded!")

In [ ]:
# ================================
# CELL 3 — Build BM25 Indexes
# ================================
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize

def build_ngrams(tokens, n):
    if n == 1:
        return tokens
    return [' '.join(tokens[i:i+n])
            for i in range(len(tokens)-n+1)]

def tokenize_ngram(text, n):
    if isinstance(text, list):
        text = ' '.join(text)
    tokens = word_tokenize(text.lower())
    return build_ngrams(tokens, n)

print("Building statute index (3-gram)...")
tokenized_statutes = [
    tokenize_ngram(t, 3)
    for t in statute_texts]
bm25_stat = BM25Okapi(
    tokenized_statutes, k1=1.6, b=0.75)
print("✅ Statute index ready!")

print("Building precedent index (5-gram)...")
tokenized_precedents = [
    tokenize_ngram(t, 5)
    for t in precedent_texts]
bm25_prec = BM25Okapi(
    tokenized_precedents, k1=1.6, b=0.75)
print("✅ Precedent index ready!")

In [ ]:
# ================================
# CELL 4 — Get BM25 Retrieved Lists
# ================================
print("Getting BM25 LSR retrieved lists...")
bm25_lsr_retrieved = []
for i, query in enumerate(query_texts):
    if i % 100 == 0:
        print(f"  Query {i}/627...")
    tq = tokenize_ngram(query, 3)
    scores = bm25_stat.get_scores(tq)
    ranked = sorted(
        range(len(scores)),
        key=lambda x: scores[x],
        reverse=True)[:10]
    bm25_lsr_retrieved.append(
        [statute_ids[idx] for idx in ranked])

print("Getting BM25 PCR retrieved lists...")
bm25_pcr_retrieved = []
for i, query in enumerate(query_texts):
    if i % 100 == 0:
        print(f"  Query {i}/627...")
    tq = tokenize_ngram(query, 5)
    scores = bm25_prec.get_scores(tq)
    ranked = sorted(
        range(len(scores)),
        key=lambda x: scores[x],
        reverse=True)[:10]
    bm25_pcr_retrieved.append(
        [precedent_ids[idx] for idx in ranked])

print("✅ BM25 lists ready!")

In [ ]:
# ================================
# CELL 5 — Ensemble + Grid Search
# ================================
import numpy as np
from sentence_transformers import util

def evaluate(all_retrieved, all_relevant, k):
    f1_scores  = []
    mrr_scores = []
    map_scores = []
    for retrieved, relevant in zip(
            all_retrieved, all_relevant):
        if len(relevant) == 0:
            continue
        p = len(set(retrieved[:k]) &
                set(relevant)) / k
        r = len(set(retrieved[:k]) &
                set(relevant)) / len(relevant)
        f1 = 2*p*r/(p+r) if p+r > 0 else 0
        f1_scores.append(f1)
        for rank, doc_id in enumerate(
                retrieved, start=1):
            if doc_id in relevant:
                mrr_scores.append(1.0/rank)
                break
        else:
            mrr_scores.append(0.0)
    return {
        f'F1@{k}'  : np.mean(f1_scores)*100,
        'MRR'      : np.mean(mrr_scores)*100
    }

def run_ensemble(bm25_retrieved,
                 corpus_embeddings,
                 corpus_ids,
                 gold_labels,
                 task_name,
                 alpha=0.5):
    all_retrieved = []
    for i, q_emb in enumerate(query_embeddings):
        sbert_scores = util.cos_sim(
            q_emb,
            corpus_embeddings)[0].cpu().numpy()
        sbert_norm = (
            (sbert_scores - sbert_scores.mean())
            / (sbert_scores.std() + 1e-8))

        bm25_scores = np.zeros(len(corpus_ids))
        for rank, doc_id in enumerate(
                bm25_retrieved[i]):
            if doc_id in corpus_ids:
                idx = corpus_ids.index(doc_id)
                bm25_scores[idx] = (
                    len(bm25_retrieved[i]) - rank)
        bm25_norm = (
            (bm25_scores - bm25_scores.mean())
            / (bm25_scores.std() + 1e-8))

        ensemble = (alpha * sbert_norm +
                   (1-alpha) * bm25_norm)
        top_idx = np.argsort(
            ensemble)[::-1][:10]
        all_retrieved.append(
            [corpus_ids[idx] for idx in top_idx])

    best_f1      = 0
    best_results = None
    for eval_k in range(1, 11):
        results = evaluate(
            all_retrieved, gold_labels, eval_k)
        if results[f'F1@{eval_k}'] > best_f1:
            best_f1      = results[f'F1@{eval_k}']
            best_results = results
    return best_results, all_retrieved

# Grid search
print("Grid search for LSR...")
lsr_ensemble = {}
for alpha in [0.0,0.1,0.2,0.3,0.4,0.5,
              0.6,0.7,0.8,0.9,1.0]:
    r, _ = run_ensemble(
        bm25_lsr_retrieved,
        statute_embeddings,
        statute_ids,
        gold_statutes,
        "LSR", alpha)
    lsr_ensemble[alpha] = r
    print(f"  alpha={alpha} → "
          f"F1={list(r.values())[0]:.2f}%")

print("\nGrid search for PCR...")
pcr_ensemble = {}
for alpha in [0.0,0.1,0.2,0.3,0.4,0.5,
              0.6,0.7,0.8,0.9,1.0]:
    r, _ = run_ensemble(
        bm25_pcr_retrieved,
        precedent_embeddings,
        precedent_ids,
        gold_precedents,
        "PCR", alpha)
    pcr_ensemble[alpha] = r
    print(f"  alpha={alpha} → "
          f"F1={list(r.values())[0]:.2f}%")

best_lsr_alpha = max(
    lsr_ensemble.keys(),
    key=lambda a: list(
        lsr_ensemble[a].values())[0])
best_pcr_alpha = max(
    pcr_ensemble.keys(),
    key=lambda a: list(
        pcr_ensemble[a].values())[0])

print(f"\n✅ Best LSR alpha: {best_lsr_alpha}")
print(f"✅ Best PCR alpha: {best_pcr_alpha}")
print("\n✅ ENSEMBLE COMPLETE!")

In [ ]:
# ================================
# CELL 6 — Save & Download
# ================================
import pickle
from google.colab import files

with open('all_results.pkl', 'wb') as f:
    pickle.dump({
        'lsr_results'    : lsr_results,
        'pcr_results'    : pcr_results,
        'lsr_ensemble'   : lsr_ensemble,
        'pcr_ensemble'   : pcr_ensemble,
        'best_lsr_alpha' : best_lsr_alpha,
        'best_pcr_alpha' : best_pcr_alpha,
    }, f)

print("✅ Saved!")
files.download('all_results.pkl')
print("✅ Downloaded!")